Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import ast
import xl
from html.parser import HTMLParser

Import data

In [5]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

RAW_PATH   = 'data/raw_data.csv' # input path
CLEAN_PATH = 'data/cleaned_data.csv' # output path

print('Imports OK')

Imports OK


Load pandas dataframe and change columns names

In [10]:
COLUMN_MAP = {
    'item key': 'product_key',
    'price': 'supplier_name',
    'flag': 'source_flag',
    'sku': 'sku',
    'attributeFamily.code': 'attribute_family',
    'category.default.title': 'category_name',
    'brand.default.title': 'brand_name',
    'rfq_only': 'rfq_only',
    'livhaven': 'livhaven_enabled',
    'mrostop': 'mrostop_enabled',
    'livhaven_hide_from_guest': 'livhaven_hide_from_guest',
    'mrostop_hide_from_guest': 'mrostop_hide_from_guest',
    'type': 'product_type',
    'status': 'status',
    'inventory_status.id': 'inventory_status',
    'primaryUnitPrecision.unit.code': 'unit_code',
    'primaryUnitPrecision.precision': 'unit_precision',
    'primaryUnitPrecision.conversionRate': 'unit_conversion_rate',
    'primaryUnitPrecision.sell': 'unit_sell',
    'names.default.fallback': 'default_name_fallback',
    'names.default.value': 'default_name',
    'names.MRO.fallback': 'mro_name_fallback',
    'names.MRO.value': 'mro_name',
    'names.LivHaven.fallback': 'livhaven_name_fallback',
    'names.LivHaven.value': 'livhaven_name',
    'shortDescriptions.default.fallback': 'default_short_description_fallback',
    'shortDescriptions.default.value': 'default_short_description',
    'shortDescriptions.MRO.fallback': 'mro_short_description_fallback',
    'shortDescriptions.MRO.value': 'mro_short_description',
    'shortDescriptions.LivHaven.fallback': 'livhaven_short_description_fallback',
    'shortDescriptions.LivHaven.value': 'livhaven_short_description',
    'descriptions.default.fallback': 'default_description_fallback',
    'descriptions.default.value': 'default_description',
    'descriptions.MRO.fallback': 'mro_description_fallback',
    'descriptions.MRO.value': 'mro_description',
    'descriptions.LivHaven.fallback': 'livhaven_description_fallback',
    'descriptions.LivHaven.value': 'livhaven_description',
    'variantFields': 'variant_fields',
    'attribute_table': 'attribute_table',
    'availability_date': 'availability_date',
    'custom_quantity_increment': 'custom_quantity_increment',
    'downloads': 'downloads',
    'featured': 'featured',
    'last_sold_price': 'last_sold_price',
    'manufacturer_description': 'manufacturer_description',
    'manufacturer_part_number': 'manufacturer_part_number',
    'newArrival': 'new_arrival',
    'livhaven_no_index': 'livhaven_no_index',
    'mrostop_no_index': 'mrostop_no_index',
    'quote_price_tolerance': 'quote_price_tolerance',
    'upc': 'upc',
    'backOrder.value': 'backorder_value',
    'category.id': 'category_id',
    'decrementQuantity.value': 'decrement_quantity_value',
    'highlightLowInventory.value': 'highlight_low_inventory_value',
    'inventoryThreshold.value': 'inventory_threshold_value',
    'lowInventoryThreshold.value': 'low_inventory_threshold_value',
    'manageInventory.value': 'manage_inventory_value',
    'maximumQuantityToOrder.value': 'maximum_quantity_to_order_value',
    'metaDescriptions.default.fallback': 'default_meta_description_fallback',
    'metaDescriptions.default.value': 'default_meta_description',
    'metaDescriptions.MRO.fallback': 'mro_meta_description_fallback',
    'metaDescriptions.MRO.value': 'mro_meta_description',
    'metaDescriptions.LivHaven.fallback': 'livhaven_meta_description_fallback',
    'metaDescriptions.LivHaven.value': 'livhaven_meta_description',
    'metaKeywords.default.fallback': 'default_meta_keywords_fallback',
    'metaKeywords.default.value': 'default_meta_keywords',
    'metaKeywords.MRO.fallback': 'mro_meta_keywords_fallback',
    'metaKeywords.MRO.value': 'mro_meta_keywords',
    'metaKeywords.LivHaven.fallback': 'livhaven_meta_keywords_fallback',
    'metaKeywords.LivHaven.value': 'livhaven_meta_keywords',
    'metaTitles.default.fallback': 'default_meta_title_fallback',
    'metaTitles.default.value': 'default_meta_title',
    'metaTitles.MRO.fallback': 'mro_meta_title_fallback',
    'metaTitles.MRO.value': 'mro_meta_title',
    'metaTitles.LivHaven.fallback': 'livhaven_meta_title_fallback',
    'metaTitles.LivHaven.value': 'livhaven_meta_title',
    'minimumQuantityToOrder.value': 'minimum_quantity_to_order_value',
    'taxCode.code': 'tax_code',
    'isUpcoming.value': 'is_upcoming_value',
    'slugPrototypes.default.fallback': 'default_slug_fallback',
    'slugPrototypes.default.value': 'default_slug',
    'slugPrototypes.MRO.fallback': 'mro_slug_fallback',
    'slugPrototypes.MRO.value': 'mro_slug',
    'slugPrototypes.LivHaven.fallback': 'livhaven_slug_fallback',
    'slugPrototypes.LivHaven.value': 'livhaven_slug',
    'itemWeight': 'item_weight'
}

DTYPE_MAP = {
    'product_key': 'Float64',
    'supplier_name': 'string',
    'source_flag': 'string',
    'sku': 'string',
    'attribute_family': 'string',
    'category_name': 'string',
    'brand_name': 'string',
    'rfq_only': 'Int8',
    'livhaven_enabled': 'Int8',
    'mrostop_enabled': 'Int8',
    'livhaven_hide_from_guest': 'Int8',
    'mrostop_hide_from_guest': 'Int8',
    'product_type': 'string',
    'status': 'string',
    'inventory_status': 'string',
    'unit_code': 'string',
    'unit_precision': 'Int8',
    'unit_conversion_rate': 'Float64',
    'unit_sell': 'Int8',
    'default_name_fallback': 'string',
    'default_name': 'string',
    'mro_name_fallback': 'string',
    'mro_name': 'string',
    'livhaven_name_fallback': 'string',
    'livhaven_name': 'string',
    'default_short_description_fallback': 'string',
    'default_short_description': 'string',
    'mro_short_description_fallback': 'string',
    'mro_short_description': 'string',
    'livhaven_short_description_fallback': 'string',
    'livhaven_short_description': 'string',
    'default_description_fallback': 'string',
    'default_description': 'string',
    'mro_description_fallback': 'string',
    'mro_description': 'string',
    'livhaven_description_fallback': 'string',
    'livhaven_description': 'string',
    'variant_fields': 'string',
    'attribute_table': 'string',
    'custom_quantity_increment': 'Float64',
    'downloads': 'string',
    'featured': 'Int8',
    'last_sold_price': 'Float64',
    'manufacturer_description': 'string',
    'manufacturer_part_number': 'string',
    'new_arrival': 'Int8',
    'livhaven_no_index': 'Int8',
    'mrostop_no_index': 'Int8',
    'quote_price_tolerance': 'Int8',
    'upc': 'string',
    'backorder_value': 'string',
    'category_id': 'Float64',
    'decrement_quantity_value': 'string',
    'highlight_low_inventory_value': 'string',
    'inventory_threshold_value': 'string',
    'low_inventory_threshold_value': 'string',
    'manage_inventory_value': 'string',
    'maximum_quantity_to_order_value': 'string',
    'default_meta_description_fallback': 'string',
    'default_meta_description': 'string',
    'mro_meta_description_fallback': 'string',
    'mro_meta_description': 'string',
    'livhaven_meta_description_fallback': 'string',
    'livhaven_meta_description': 'string',
    'default_meta_keywords_fallback': 'string',
    'default_meta_keywords': 'string',
    'mro_meta_keywords_fallback': 'string',
    'mro_meta_keywords': 'string',
    'livhaven_meta_keywords_fallback': 'string',
    'livhaven_meta_keywords': 'string',
    'default_meta_title_fallback': 'string',
    'default_meta_title': 'string',
    'mro_meta_title_fallback': 'string',
    'mro_meta_title': 'string',
    'livhaven_meta_title_fallback': 'string',
    'livhaven_meta_title': 'string',
    'minimum_quantity_to_order_value': 'string',
    'tax_code': 'string',
    'is_upcoming_value': 'string',
    'default_slug_fallback': 'string',
    'default_slug': 'string',
    'mro_slug_fallback': 'string',
    'mro_slug': 'string',
    'livhaven_slug_fallback': 'string',
    'livhaven_slug': 'string',
    'item_weight': 'Float64'
}

data = pd.read_csv(RAW_PATH, dtype=str)
data = data.rename(columns=COLUMN_MAP)

valid_dtype_map = {k: v for k, v in DTYPE_MAP.items() if k in data.columns}
data = data.astype(valid_dtype_map)

if 'availability_date' in data.columns:
    data['availability_date'] = pd.to_datetime(data['availability_date'], errors='coerce')

In [12]:
# Full null audit — save so you can refer back to it
null_audit = (
    pd.DataFrame({
        'pct_null':   (data.isna().mean() * 100).round(2),
        'null_count': data.isna().sum(),
        'dtype':      data.dtypes.astype(str),
        'n_unique':   data.nunique(),
    })
    .sort_values('pct_null', ascending=False)
    .reset_index()
    .rename(columns={'index': 'column'})
)

null_audit.to_csv('data/null_audit.csv', index=False)
print('Null audit saved → data/null_audit.csv')
null_audit.head(20)

Null audit saved → data/null_audit.csv


,column,pct_null,null_count,dtype,n_unique
0,default_description_fallback,100.00,292658,string,0
1,default_meta_title_fallback,100.00,292658,string,0
2,variant_fields,100.00,292658,string,0
3,mro_meta_keywords,100.00,292658,string,0
4,mro_short_description,100.00,292647,string,11
5,default_meta_keywords_fallback,100.00,292658,string,0
6,default_short_description_fallback,100.00,292658,string,0
7,mro_name,100.00,292658,string,0
8,livhaven_meta_keywords,100.00,292658,string,0
9,availability_date,100.00,292658,datetime64[ns],0


Drop columns with large amounts of NA and unimportant information

In [13]:
cols_to_drop = [
    'default_description_fallback',
    'default_meta_title_fallback',
    'variant_fields',
    'mro_meta_keywords',
    'mro_short_description',
    'default_meta_keywords_fallback',
    'default_short_description_fallback',
    'mro_name',
    'livhaven_meta_keywords',
    'availability_date',
    'default_name_fallback',
    'default_meta_description_fallback',
    'mro_slug',
    'default_slug_fallback',
    'default_meta_keywords',
    'decrement_quantity_value',
    'highlight_low_inventory_value',
    'inventory_threshold_value',
    'maximum_quantity_to_order_value',
    'low_inventory_threshold_value',
    'manage_inventory_value',
    'backorder_value',
    'is_upcoming_value',
    'mro_meta_keywords_fallback',
    'custom_quantity_increment',
    'minimum_quantity_to_order_value',
    'upc'
]

data = data.drop(columns=cols_to_drop, errors='ignore')

Standardize text columns

In [ ]:
lower_and_strip_cols = [
    'supplier_name',
    'category_name',
    'brand_name',
    'default_name',
    'default_short_description',
    'default_description',
    'manufacturer_description',
    'default_meta_description',
    'default_meta_title'
]

strip_only_cols = [
    'sku',
    'manufacturer_part_number',
    'attribute_family',
    'inventory_status',
    'unit_code',
    'tax_code'
]

for col in lower_and_strip_cols:
    if col in data.columns:
        data[col] = data[col].astype('string').str.strip().str.lower()

for col in strip_only_cols:
    if col in data.columns:
        data[col] = data[col].astype('string').str.strip()